# Black-box source--target scatter

Generate a scatter where each point has outgoing score as source and incoming score as target.

# Black-box transfer analysis configuration

All three notebooks use the same `TransferViewConfig` object, so the attack matrix, source--target scatter, and top-k tables are directly comparable.

Key knobs:

- `source_grain` / `target_grain`: `family`, `architecture`, or `variant`.
- `comparison_mode`:
  - `inter_variant`: exclude exact same variant only.
  - `inter_architecture`: exclude same architecture.
  - `inter_model`: exclude same family.
- `variant_aggregate_mode`: how variants are collapsed into architectures/families: `mean`, `median`, `best_damage`, `best_robust`.
- `architecture_aggregate_mode`: how architectures are collapsed into families: `mean`, `median`, `best_damage`, `best_robust`.
- `split_ansatz_as_architecture`: when `True`, VQM/HVQL/etc. ansatz variants become architecture labels such as `VQM-Strong` or `HVQL-Random`.
- `blacklist_contains` and `model_blacklist` apply to all outputs.


In [ ]:

from pathlib import Path
import sys

try:
    from analytics.paths import find_project_root, output_dir
except ModuleNotFoundError:
    # Allows running the notebook from a copied location.
    repo_root = Path.cwd()
    while repo_root != repo_root.parent and not (repo_root / "src").exists():
        repo_root = repo_root.parent
    sys.path.insert(0, str(repo_root / "src"))
    from analytics.paths import find_project_root, output_dir

from analytics.attacks.transfer_role_tools import (
    TransferViewConfig,
    VIEW_TEMPLATES,
    describe_config,
    get_template,
    list_templates,
    load_or_build_pair_metrics,
    build_attack_matrix,
    build_source_target_scatter,
    build_top_sources_by_target,
    build_top_targets_by_source,
    plot_attack_matrix,
    plot_source_target_scatter,
    save_figure,
    write_outputs,
)

ROOT = find_project_root()
PAIR_METRICS_CACHE = output_dir(ROOT, "attack_transfer_role_tables") / "pair_metrics.csv"
OUT_DIR = output_dir(ROOT, "blackbox_transfer_views")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Template shortcuts:
# list_templates()
TEMPLATE_NAME = "architecture_mean"

CONFIG = get_template(
    TEMPLATE_NAME,
    source_grain="architecture",
    target_grain="architecture",
    comparison_mode="inter_architecture",
    metric="Transfer ASR",
    top_k=3,
    variant_aggregate_mode="mean",
    architecture_aggregate_mode="mean",
    split_ansatz_as_architecture=False,
    # Exact model names to exclude.
    model_blacklist=tuple(),
    # Substring rules, useful for quick exclusions such as ("HQuanv",).
    blacklist_contains=tuple(),
)

display(describe_config(CONFIG))


In [ ]:
display(list_templates())

In [ ]:

pair_df = load_or_build_pair_metrics(
    pair_metrics_cache=PAIR_METRICS_CACHE,
    rebuild=False,
)
print(f"Loaded pair metrics: {pair_df.shape[0]:,} rows × {pair_df.shape[1]:,} columns")


In [ ]:

profile = build_source_target_scatter(pair_df, CONFIG)
display(profile)

prefix = f"scatter_{CONFIG.source_grain}_to_{CONFIG.target_grain}_{CONFIG.comparison_mode}_{CONFIG.metric.replace(' ', '_')}"
outputs = write_outputs(OUT_DIR, prefix=prefix, scatter_profile=profile)
outputs


In [ ]:

fig, ax = plot_source_target_scatter(
    profile,
    title=(
        f"Source attack potential vs target vulnerability — {CONFIG.metric} | "
        f"{CONFIG.source_grain}, {CONFIG.comparison_mode}, "
        f"variant agg={CONFIG.variant_aggregate_mode}"
    ),
    annotate=True,
)
fig_path = save_figure(fig, OUT_DIR / f"{prefix}.png")
fig_path
